# Mini Project
### Research Question: **Does local sequential information improve supervised topic classification on the Reuters corpus compared to bag-of-words features?**

In text classification, documents can be represented through different types of features. One common approach is the bag-of-words model, where texts are represented through individual words without considering their order. Another approach includes local sequential information, such as bigrams and trigrams, which preserve short word sequences and nearby contextual patterns. However, larger sequential representations may also increase sparsity, since many word combinations occur less frequently in the data.

This project investigates whether local sequential information improves supervised topic classification on the Reuters corpus compared to traditional bag-of-words features. More specifically, the project compares unigram, bigram, trigram, combined unigram + bigram, and combined unigram + bigram + trigram representations using the same Naïve Bayes classifier.

In [9]:
import nltk
# nltk.download('reuters')
# nltk.download('stopwords')
from nltk import bigrams, trigrams
from nltk.corpus import stopwords
from nltk.corpus import reuters
import string
import random
from collections import Counter

### Data Prefiltering and Partitioning

Keep only documents associated with exactly one topic.

In [2]:
single_topic_docs = [
    (fileid, reuters.categories(fileid)[0])
    for fileid in reuters.fileids()
    if len(reuters.categories(fileid)) == 1
]

print("Documents after single-topic filtering:", len(single_topic_docs))
print("example:", single_topic_docs[:5])


Documents after single-topic filtering: 9160
example: [('test/14826', 'trade'), ('test/14828', 'grain'), ('test/14839', 'ship'), ('test/14842', 'gold'), ('test/14843', 'acq')]


Calculate topic frequencies and select the 30 most frequent categories.

In [3]:
topic_counts = Counter(topic for fileid, topic in single_topic_docs)
print(topic_counts)

top_30_topics = {topic for topic, count in topic_counts.most_common(30)}

print("30 most frequent topics:")
print(sorted(top_30_topics))

Counter({'earn': 3923, 'acq': 2292, 'crude': 374, 'trade': 326, 'money-fx': 309, 'interest': 272, 'money-supply': 151, 'ship': 144, 'sugar': 122, 'coffee': 112, 'gold': 90, 'gnp': 74, 'cpi': 71, 'cocoa': 61, 'grain': 51, 'alum': 50, 'jobs': 49, 'reserves': 49, 'ipi': 45, 'copper': 44, 'rubber': 40, 'iron-steel': 38, 'nat-gas': 36, 'bop': 31, 'veg-oil': 30, 'tin': 27, 'cotton': 24, 'wpi': 23, 'livestock': 22, 'orange': 22, 'retail': 20, 'pet-chem': 19, 'gas': 18, 'housing': 17, 'strategic-metal': 15, 'lei': 14, 'lumber': 14, 'zinc': 13, 'fuel': 11, 'meal-feed': 11, 'carcass': 11, 'income': 11, 'heat': 10, 'oilseed': 9, 'lead': 8, 'yen': 6, 'dlr': 6, 'instal-debt': 6, 'potato': 5, 'tea': 5, 'nickel': 4, 'cpu': 4, 'silver': 4, 'platinum': 3, 'jet': 3, 'groundnut': 2, 'rice': 1, 'hog': 1, 'naphtha': 1, 'propane': 1, 'coconut': 1, 'nzdlr': 1, 'dmk': 1, 'l-cattle': 1, 'rand': 1})
30 most frequent topics:
['acq', 'alum', 'bop', 'cocoa', 'coffee', 'copper', 'cotton', 'cpi', 'crude', 'earn', 'g

Remove documents belonging to less frequent topics.

In [4]:
filtered_docs = [
    (fileid, topic)
    for fileid, topic in single_topic_docs
    if topic in top_30_topics
]

print("Documents after top-30 topic filtering:", len(filtered_docs))

Documents after top-30 topic filtering: 8902


Separate the predefined Reuters training and test partitions.

In [5]:
train_docs = [
    (fileid, topic)
    for fileid, topic in filtered_docs
    if fileid.startswith("training/")
]

test_docs = [
    (fileid, topic)
    for fileid, topic in filtered_docs
    if fileid.startswith("test/")
]

print("Predefined training documents:", len(train_docs))
print("Final test documents:", len(test_docs))

Predefined training documents: 6407
Final test documents: 2495


Randomly split the predefined training partition into an actual training set and a development set.

In [69]:
random.seed(42) #22 #30
random.shuffle(train_docs)

split_index = int(len(train_docs) * 0.9)

new_train_docs = train_docs[:split_index]
dev_docs = train_docs[split_index:] 

print("Actual training documents:", len(new_train_docs))
print("Development documents:", len(dev_docs))
print("Final test documents:", len(test_docs))

Actual training documents: 5766
Development documents: 641
Final test documents: 2495


Inspect the topic distributions across the different partitions.

In [58]:
train_distribution = Counter(topic for fileid, topic in new_train_docs)
dev_distribution = Counter(topic for fileid, topic in dev_docs)
test_distribution = Counter(topic for fileid, topic in test_docs)

print("Most common topics in the actual training set:")
print(train_distribution.most_common(10))

print("\nMost common topics in the development set:")
print(dev_distribution.most_common(10))

print("\nMost common topics in the final test set:")
print(test_distribution.most_common(10))

Most common topics in the actual training set:
[('earn', 2551), ('acq', 1443), ('trade', 227), ('crude', 226), ('money-fx', 199), ('interest', 170), ('money-supply', 114), ('ship', 93), ('sugar', 83), ('coffee', 82)]

Most common topics in the development set:
[('earn', 289), ('acq', 153), ('crude', 27), ('money-fx', 23), ('trade', 23), ('interest', 21), ('ship', 15), ('sugar', 14), ('money-supply', 9), ('coffee', 8)]

Most common topics in the final test set:
[('earn', 1083), ('acq', 696), ('crude', 121), ('money-fx', 87), ('interest', 81), ('trade', 76), ('ship', 36), ('money-supply', 28), ('sugar', 25), ('coffee', 22)]


The topic distributions across the training, development, and final test sets are relatively similar. The dataset is clearly imbalanced, since topics such as “earn” and “acq” are much more frequent than the others in all partitions. This imbalance may influence classification performance, potentially favoring the most frequent categories.

### Most Frequent Class Baseline

Before training the classifiers, a simple baseline was calculated using the most frequent class in the training data. 

The baseline classifier always predicts the same topic, regardless of the document content. This provides a reference point for evaluating whether the machine learning models actually learn meaningful patterns from the text.

Since the Reuters corpus is imbalanced, comparing the classifiers against this baseline is important to better interpret the final results.

In [59]:
most_frequent_topic = Counter(topic for fileid, topic in train_docs).most_common(1)[0][0]

baseline_accuracy = sum(
    1 for fileid, topic in test_docs
    if topic == most_frequent_topic
) / len(test_docs)

print("Most frequent class:", most_frequent_topic)
print(f"Most frequent class baseline accuracy: {baseline_accuracy:.4f}")

Most frequent class: earn
Most frequent class baseline accuracy: 0.4341


### Preprocessing

A preprocessing function is defined for the experiments. In the main setup, only lowercasing is applied in order to preserve as much of the original textual structure as possible, including punctuation, stopwords, and local syntactic sequences. This choice is particularly important for the analysis of bigram features, since removing tokens could alter the original sequential structure of the Reuters documents.

Additional preprocessing steps, such as non-alphabetic filtering and stopword removal, are included in the code as optional commented lines. These will later be activated in a separate comparison experiment in order to evaluate how noise reduction affects classification performance and the interpretation of sequential information.

This comparison makes it possible to distinguish between preserving the original local context of the documents and emphasizing only content-bearing lexical information.

In [60]:
stop_words = set(stopwords.words("english"))

def preprocess(document):
    return [
        w.lower()
        for w in document
        #if w.isalpha()                     # <-- Uncomment to enable preprocessing
        #and w.lower() not in stop_words    # <-- Uncomment to enable preprocessing
    ]

### Feature Extraction

The unigram, bigram, and trigram feature lists are created from the actual training set only. This avoids using information from the development or final test sets during feature selection and helps prevent data leakage during the experiments.

In [61]:
all_words = nltk.FreqDist(
    w.lower()
    for fileid, topic in new_train_docs
    for w in preprocess(reuters.words(fileid))
)

word_features = [w for w, f in all_words.most_common(1000)]

all_bigrams = nltk.FreqDist(
    bigram
    for fileid, topic in new_train_docs
    for bigram in bigrams(preprocess(reuters.words(fileid)))
)

bigram_features_list = [bg for bg, f in all_bigrams.most_common(1000)]

all_trigrams = nltk.FreqDist(
    trigram
    for fileid, topic in new_train_docs
    for trigram in trigrams(preprocess(reuters.words(fileid)))
)

trigram_features_list = [tg for tg, f in all_trigrams.most_common(1000)]

print('Example of "word features":', word_features[:5])
print('Example of "bigram features":', bigram_features_list[:5])
print('Example of "trigram features":', trigram_features_list[:5])

Example of "word features": ['.', ',', 'the', 'of', 'to']
Example of "bigram features": [(',', '000'), ('lt', ';'), ('&', 'lt'), ("'", 's'), ('.', 'the')]
Example of "trigram features": [('&', 'lt', ';'), ('u', '.', 's'), ('.', 's', '.'), (',', '000', 'vs'), (',', '000', 'dlrs')]


Define unigram, bigram, trigram and combined feature extractors.

In [62]:
def contains_features(document): 
    document_words = set(preprocess(document)) 
    features = {}

    for word in word_features:
        features[f'contains({word})'] = (word in document_words)

    return features


def bigram_features(document):
    document_bigrams = set(bigrams(preprocess(document)))
    features = {}

    for w1, w2 in bigram_features_list:
        features[f'bigram({w1} {w2})'] = ((w1, w2) in document_bigrams)

    return features


def trigram_features(document):
    document_trigrams = set(trigrams(preprocess(document)))
    features = {}

    for w1, w2, w3 in trigram_features_list:
        features[f'trigram({w1} {w2} {w3})'] = ((w1, w2, w3) in document_trigrams)

    return features


def combined_unigram_bigram_features(document):
    features = {}

    features.update(contains_features(document))
    features.update(bigram_features(document))

    return features


def combined_all_features(document):
    features = {}

    features.update(contains_features(document))
    features.update(bigram_features(document))
    features.update(trigram_features(document))

    return features

### Development Set Evaluation

This function trains a Naïve Bayes classifier on the actual training set and evaluates it on the development set. It is used to compare the different feature extraction methods before the final test evaluation.

In [41]:
def evaluate_feature_extractor(feature_function, feature_name):
    train_set = [
        (feature_function(reuters.words(fileid)), topic)
        for fileid, topic in new_train_docs
    ]

    dev_set = [
        (feature_function(reuters.words(fileid)), topic)
        for fileid, topic in dev_docs
    ]

    classifier = nltk.NaiveBayesClassifier.train(train_set)
    accuracy = nltk.classify.accuracy(classifier, dev_set)

    print(f"Feature Extraction: {feature_name}")
    print(f"Development accuracy: {accuracy:.4f}")
    print("30 most informative features:")
    classifier.show_most_informative_features(30)
    print()

    return classifier

The five feature extraction methods are tested on the development set: unigram features, bigram features, trigram features, combined unigram + bigram features, and combined unigram + bigram + trigram features.

In [42]:
unigram_classifier = evaluate_feature_extractor(contains_features, "unigram features")

bigram_classifier = evaluate_feature_extractor(bigram_features, "bigram features")

trigram_classifier = evaluate_feature_extractor(trigram_features,"trigram features")

combined_unigram_bigram_classifier = evaluate_feature_extractor(combined_unigram_bigram_features, "combined unigram + bigram features")

combined_all_classifier = evaluate_feature_extractor(combined_all_features, "combined unigram + bigram + trigram features")

Feature Extraction: unigram features
Development accuracy: 0.7972
30 most informative features:
Most Informative Features
        contains(coffee) = True           coffee : earn   =   1685.1 : 1.0
        contains(copper) = True           copper : earn   =   1000.8 : 1.0
        contains(rubber) = True           rubber : earn   =   1000.2 : 1.0
   contains(agriculture) = True           livest : earn   =    908.2 : 1.0
         contains(grain) = True            grain : earn   =    857.4 : 1.0
       contains(council) = True            cocoa : earn   =    847.7 : 1.0
         contains(sugar) = True            sugar : earn   =    704.9 : 1.0
         contains(index) = True              wpi : earn   =    690.2 : 1.0
     contains(inflation) = True              cpi : acq    =    676.5 : 1.0
        contains(quotas) = True           coffee : acq    =    554.7 : 1.0
          contains(port) = True             ship : earn   =    551.0 : 1.0
          contains(pact) = True           rubber : ea

The development results show that unigram features achieved the best performance on the development set (0.7972), followed closely by the combined unigram + bigram model (0.7894). Bigram features alone obtained lower accuracy (0.7629), while trigram features performed worst (0.7207).

The unigram model is already strong because Reuters topic classification is highly keyword-driven, with informative lexical items such as `copper`, `grain`, `sugar`, and `inflation`.

Bigram features capture meaningful local expressions such as `trade deficit`, `federal reserve`, and `prime rate`, but sequential information alone is not sufficient for the task. Trigram features further increase sparsity and often capture overly specific sequences containing numbers and punctuation.

Overall, the results suggest that short local sequential information can be useful when combined with unigram features, while larger n-grams introduce more noise than useful contextual information.

### Evaluation Metrics

Accuracy alone is not always sufficient for evaluating text classification systems, especially on imbalanced datasets such as Reuters. 

For this reason, precision, recall, and F1-score are also calculated for selected categories in order to better understand the strengths and weaknesses of the classifiers.

- *Precision* measures how many documents predicted as a given topic were actually correct. 

- *Recall* measures how many documents belonging to a topic were successfully identified by the classifier.

- *F1-score* combines precision and recall into a single metric and provides a more balanced evaluation, especially useful for imbalanced datasets such as Reuters.

In [16]:
def precision_recall_f1(gold, predicted):
    labels = sorted(set(gold))
    
    gold_counts = Counter(gold)
    predicted_counts = Counter(predicted)
    
    results = {}
    
    for label in labels:
        true_positive = sum(1 for g, p in zip(gold, predicted) if g == label and p == label)
        
        precision = (true_positive / predicted_counts[label] if predicted_counts[label] > 0 else 0)
        recall = (true_positive / gold_counts[label] if gold_counts[label] > 0 else 0) 
        f1 = (2 * precision * recall / (precision + recall) if precision + recall > 0 else 0)

        results[label] = (precision, recall, f1)
    return results

### Final Test Evaluation

This function performs the final evaluation. The classifier is trained on the full predefined Reuters training partition and tested on the held-out Reuters test partition. It reports accuracy, selected precision/recall/F1 scores, optional confusion matrix output, and informative features.

In [63]:
def final_evaluation(feature_function, feature_name, show_confusion=False, show_features=True):

    train_set = [
        (feature_function(reuters.words(fileid)), topic)
        for fileid, topic in train_docs
    ]

    test_set = [
        (feature_function(reuters.words(fileid)), topic)
        for fileid, topic in test_docs
    ]

    classifier = nltk.NaiveBayesClassifier.train(train_set)

    accuracy = nltk.classify.accuracy(classifier, test_set)

    print(f"Final evaluation using: {feature_name}")
    print(f"Final test accuracy: {accuracy:.4f}")
    print()

    gold = [topic for features, topic in test_set]
    predicted = classifier.classify_many([features for features, topic in test_set])
    scores = precision_recall_f1(gold, predicted)

    print("Precision, recall and F1-score for selected topics:")
    for label in ["earn", "acq", "crude", "trade", "interest", "money-fx"]:
        precision, recall, f1 = scores[label]
        print(f"{label:12s} precision: {precision:.3f} recall: {recall:.3f} f1: {f1:.3f}")
    print()

    if show_confusion:
        cm = nltk.ConfusionMatrix(gold, predicted)
        print(cm)
        print()

    if show_features:
        print("20 most informative features:")
        classifier.show_most_informative_features(20)

    return classifier

The unigram model is evaluated on the final test set.

In [ ]:
final_unigram_classifier = final_evaluation(contains_features,"unigram features", show_confusion=True, show_features=True)

Final evaluation using: unigram features
Final test accuracy: 0.8060

Precision, recall and F1-score for selected topics:
earn         precision: 0.913 recall: 0.934 f1: 0.923
acq          precision: 0.897 recall: 0.841 f1: 0.868
crude        precision: 0.645 recall: 0.570 f1: 0.605
trade        precision: 0.582 recall: 0.750 f1: 0.655
interest     precision: 0.800 recall: 0.543 f1: 0.647
money-fx     precision: 0.737 recall: 0.644 f1: 0.687



The bigram model is evaluated on the final test set.

In [ ]:
final_bigram_classifier = final_evaluation(bigram_features,"bigram features", show_confusion=False, show_features=True)

Final evaluation using: bigram features
Final test accuracy: 0.7739

Precision, recall and F1-score for selected topics:
earn         precision: 0.992 recall: 0.910 f1: 0.949
acq          precision: 0.912 recall: 0.884 f1: 0.898
crude        precision: 0.613 recall: 0.537 f1: 0.573
trade        precision: 0.456 recall: 0.684 f1: 0.547
interest     precision: 0.361 recall: 0.654 f1: 0.465
money-fx     precision: 0.580 recall: 0.586 f1: 0.583



The trigram model is evaluated on the final test set.

In [ ]:
final_trigram_classifier = final_evaluation(trigram_features,"trigram features", show_confusion=False, show_features=True)

Final evaluation using: trigram features
Final test accuracy: 0.7222

Precision, recall and F1-score for selected topics:
earn         precision: 0.999 recall: 0.886 f1: 0.939
acq          precision: 0.834 recall: 0.861 f1: 0.847
crude        precision: 0.495 recall: 0.413 f1: 0.450
trade        precision: 0.400 recall: 0.763 f1: 0.525
interest     precision: 0.455 recall: 0.432 f1: 0.443
money-fx     precision: 0.354 recall: 0.322 f1: 0.337



The combined unigram + bigram model is evaluated on the final test set.

In [ ]:
final_combined_u_b_classifier = final_evaluation(combined_unigram_bigram_features,"combined unigram + bigram features", show_confusion=False, show_features=True)

Final evaluation using: combined unigram + bigram features
Final test accuracy: 0.7948

Precision, recall and F1-score for selected topics:
earn         precision: 0.946 recall: 0.922 f1: 0.934
acq          precision: 0.910 recall: 0.861 f1: 0.885
crude        precision: 0.676 recall: 0.570 f1: 0.619
trade        precision: 0.527 recall: 0.763 f1: 0.624
interest     precision: 0.719 recall: 0.568 f1: 0.634
money-fx     precision: 0.697 recall: 0.609 f1: 0.650



The combined unigram + bigram + trigram model is evaluated on the final test set.

In [ ]:
final_combined_classifier = final_evaluation(combined_all_features,"combined features", show_confusion=False, show_features=True)

Final evaluation using: combined features
Final test accuracy: 0.7948

Precision, recall and F1-score for selected topics:
earn         precision: 0.980 recall: 0.922 f1: 0.951
acq          precision: 0.912 recall: 0.876 f1: 0.894
crude        precision: 0.622 recall: 0.570 f1: 0.595
trade        precision: 0.514 recall: 0.737 f1: 0.605
interest     precision: 0.557 recall: 0.605 f1: 0.580
money-fx     precision: 0.675 recall: 0.598 f1: 0.634



### Results and Discussion

The final results show that unigram features achieved the best overall performance on the Reuters test set, with an accuracy of 0.8028. The combined unigram + bigram model obtained a very similar result (0.7968), while bigram features alone performed worse (0.7768). Trigram features achieved the lowest accuracy (0.7210), and adding trigrams to the combined representation did not improve the results.

| Feature extraction | Final test accuracy |
|---|---:|
| Unigram | 0.8028 |
| Bigram | 0.7768 |
| Trigram | 0.7210 |
| Combined unigram + bigram | 0.7968 |
| Combined unigram + bigram + trigram | 0.7956 |

The precision, recall and F1-scores confirm this trend. Unigram features achieve the most stable performance across the selected topics, while trigram features perform considerably worse on categories such as `interest`, `money-fx`, and `crude`.

The unigram model is already very strong because Reuters topic classification is largely keyword-driven. Many of the most informative features are highly topic-specific lexical items such as `coffee`, `copper`, `rubber`, `grain`, `inflation`, and `sugar`.

Bigram features capture meaningful local expressions such as `trade deficit`, `federal reserve`, `prime rate`, and `european community`. However, the lower performance of the bigram-only model suggests that local sequential information alone is not sufficient for accurate topic classification.

The trigram model further increases sparsity and often captures overly specific sequences containing punctuation, numbers, and fragmented expressions such as `rose 0 .`, `0 . 4`, and `in february ,`. This makes many trigram features too rare or too specific to generalize effectively across documents.

Overall, the results suggest that local sequential information can contribute useful contextual information, but Reuters topic classification remains primarily driven by informative lexical content and recurring economic collocations rather than broader sequential structure.

The **confusion matrix** for the unigram model shows that the unigram model performs very well on the largest classes, especially `earn` and `acq`. However, several errors are directed toward these frequent categories, especially toward `earn`, confirming that class imbalance influences the classifier.

A clear pattern also appears around financial and monetary topics. Categories such as `interest`, `money-fx`, and `money-supply` are often confused with one another. In particular, the column for `money-supply` attracts several errors from related financial categories, suggesting that monetary vocabulary overlaps across different Reuters topics.

Another relevant pattern concerns `ship`. Some documents from trade- and commodity-related categories are classified as `ship`, probably because they share vocabulary connected to exports, transport, ports, and commercial exchange.

Overall, the matrix suggests that errors are not random. They mainly occur between semantically close topics or toward very frequent classes. This supports the idea that the model relies heavily on lexical cues: when different topics share similar vocabulary, the classifier becomes more uncertain.

### Comparison with full preprocessing

An additional comparison was carried out using full preprocessing, where non-alphabetic tokens and English stopwords were removed before feature extraction. This was done to check whether reducing noise in the feature space could improve the models, especially those based on n-gram features.

| Feature extraction | Accuracy without preprocessing | Accuracy with preprocessing |
|---|---:|---:|
| Unigram | 0.8028 | 0.8465 |
| Bigram | 0.7768 | 0.8265 |
| Trigram | 0.7210 | 0.7335 |
| Combined unigram + bigram | 0.7968 | 0.8641 |
| Combined unigram + bigram + trigram | 0.7956 | 0.8537 |

The results show that preprocessing improves all models, but not equally. The improvement is especially clear for unigrams, bigrams, and the combined unigram + bigram model. Trigrams improve only slightly, suggesting that longer sequences remain sparse and harder to generalize even after noise reduction.

The bigram model benefits strongly from preprocessing because it captures more meaningful multiword expressions and economic collocations, such as `current account`, `industrial production`, `federal reserve`, and `prime rate`. These expressions behave almost like semantic units in the Reuters corpus and are highly informative for topic classification.

The combined unigram + bigram model achieves the best overall accuracy. This suggests that Reuters topic classification is strongly keyword-driven, but bigrams add useful information by capturing domain-specific multiword expressions that single-word features may miss.

The precision, recall, and F1-scores also support this interpretation. With preprocessing, the combined unigram + bigram model improves several selected categories compared to unigrams alone, especially `crude` (F1: 0.720 -> 0.829) and `interest` (0.645 -> 0.733). This shows that bigram information does not only improve overall accuracy, but also helps some economically related categories.

Adding trigrams does not further improve the model. Although some trigram features are meaningful, such as `gross domestic product` and `barrels per day`, many trigram sequences are still too specific. This increases sparsity and slightly reduces performance compared to the unigram + bigram combination.

Overall, the comparison suggests that the most useful form of sequential information in this task is not a broader syntactic sequence, but short domain-specific multiword expressions. These are best captured by bigrams and become most effective when combined with unigram keyword features.

### Conclusion

The experiments suggest that the forms of sequential information tested in this project do not generally improve supervised topic classification on the Reuters corpus. Unigram features remain a very strong representation, confirming that this task is largely keyword-driven.

The main exception is the combined unigram + bigram model after preprocessing, which achieved the best final test accuracy. This result suggests that bigrams can be useful when the feature space is cleaned, mainly because they capture recurring economic collocations and multiword expressions such as `current account`, `industrial production`, `federal reserve`, and `trade deficit`. In this sense, the improvement seems to come less from broad sequential structure and more from short domain-specific lexical units.

Trigram features did not improve performance. This is probably due to sparsity: longer sequences occur less frequently and are harder for the classifier to generalize from. Adding trigrams to the combined model also did not improve the results, confirming that larger local contexts are not necessarily more useful for this task.

A limitation of this project is that the experiments were not averaged over many repeated runs. More reliable results could be obtained by repeating the experiments several times with different random seeds and averaging the scores. However, this would require much more computational time, since some models, especially trigram-based and combined feature models, take several minutes to run.

To partially check the stability of the results, I repeated the experiments with two additional random seeds, 30 and 22. The same general trend was observed: unigram features remained the strongest representation without preprocessing, trigram features consistently performed worst, and the combined models did not clearly outperform unigrams in the non-preprocessed setting. This suggests that the main trend is reasonably stable, although a larger number of repeated runs would be needed for stronger conclusions.